<a href="https://colab.research.google.com/github/LucasAlegre/vote-network/blob/master/vote_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install and import libraries (need to restart the session after install)

In [1]:
! pip install python-igraph
# Módulo de visualização (Sigma.js + Graphology). No Colab, baixa do repositório.
import os
if not os.path.exists('graph_viz.py'):
    ! wget -q https://raw.githubusercontent.com/LucasAlegre/vote-network/master/graph_viz.py

In [2]:
import pandas as pd
import urllib.request
import urllib.error
import random
import time
import importlib
import requests
from datetime import date
import numpy as np
import igraph as ig
from igraph import Graph, summary
from itertools import combinations
import os
import graph_viz
importlib.reload(graph_viz)  # relê o módulo se o arquivo mudou (sem reiniciar o kernel)

<module 'graph_viz' from '/home/lucas/vote-network/graph_viz.py'>

## Data Retrieval and Processing

In [3]:
# Períodos da linha do tempo: cada um vira um quadro do grafo (um por tema e um "Geral").
date_intervals = [(f'{ano}-01-01', f'{ano}-12-31') for ano in range(2022, 2027)]
# date_intervals = [('2022-02-01', '2026-01-31')]  # um único período (sem linha do tempo)

def period_label(ini, fim):
    """'2023' para um ano civil inteiro; senão '01/02/2022–31/01/2026'."""
    ini, fim = pd.to_datetime(ini), pd.to_datetime(fim)
    if ini.year == fim.year and (ini.month, ini.day) == (1, 1) and (fim.month, fim.day) == (12, 31):
        return str(ini.year)
    return f'{ini:%d/%m/%Y}–{fim:%d/%m/%Y}'

# {rótulo: (início, fim)}, com o fim exclusivo para incluir o último dia inteiro
periods = {period_label(ini, fim): (pd.to_datetime(ini), pd.to_datetime(fim) + pd.Timedelta(days=1))
           for ini, fim in date_intervals}
start_date = min(ini for ini, _ in periods.values())
end_date = max(fim for _, fim in periods.values())
periods

{'2022': (Timestamp('2022-01-01 00:00:00'), Timestamp('2023-01-01 00:00:00')),
 '2023': (Timestamp('2023-01-01 00:00:00'), Timestamp('2024-01-01 00:00:00')),
 '2024': (Timestamp('2024-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00')),
 '2025': (Timestamp('2025-01-01 00:00:00'), Timestamp('2026-01-01 00:00:00')),
 '2026': (Timestamp('2026-01-01 00:00:00'), Timestamp('2027-01-01 00:00:00'))}

In [4]:
# Baixa (uma vez, em data/) e lê os CSVs anuais dos dados abertos da Câmara
os.makedirs('data', exist_ok=True)
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.9.0.7) Gecko/2009021910 Firefox/3.0.7'}

def download(kind, year):
    path = f'data/{kind}-{year}.csv'
    if not os.path.isfile(path):
        url = f'https://dadosabertos.camara.leg.br/arquivos/{kind}/csv/{kind}-{year}.csv'
        with urllib.request.urlopen(urllib.request.Request(url, None, HEADERS)) as response:
            data = response.read()
        with open(path + '.part', 'wb') as f:  # só vira o arquivo final se o download terminar
            f.write(data)
        os.replace(path + '.part', path)
    return pd.read_csv(path, sep=';', low_memory=False)

years = range(start_date.year, (end_date - pd.Timedelta(days=1)).year + 1)

# Votos nominais (sem votações simbólicas) dentro dos períodos
votes = pd.concat([download('votacoesVotos', year) for year in years], ignore_index=True)
votes = votes[votes['voto'] != 'Simbólico'].copy()
votes['dataHoraVoto'] = pd.to_datetime(votes['dataHoraVoto'])
votes = votes[(votes['dataHoraVoto'] >= start_date) & (votes['dataHoraVoto'] < end_date)]
votes = votes.sort_values('dataHoraVoto', ignore_index=True)
votes

,idVotacao,uriVotacao,dataHoraVoto,voto,deputado_id,deputado_uri,deputado_nome,deputado_siglaPartido,deputado_uriPartido,deputado_siglaUf,deputado_idLegislatura,deputado_urlFoto
0,2312874-9,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09 16:43:57,Artigo 17,160541,https://dadosabertos.camara.leg.br/api/v2/depu...,Arthur Lira,PP,https://dadosabertos.camara.leg.br/api/v2/part...,AL,56,https://www.camara.leg.br/internet/deputado/ba...
1,2312874-9,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09 16:44:11,Não,178972,https://dadosabertos.camara.leg.br/api/v2/depu...,Alex Manente,CIDADANIA,https://dadosabertos.camara.leg.br/api/v2/part...,SP,56,https://www.camara.leg.br/internet/deputado/ba...
2,2312874-9,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09 16:44:12,Sim,178871,https://dadosabertos.camara.leg.br/api/v2/depu...,Evair Vieira de Melo,PP,https://dadosabertos.camara.leg.br/api/v2/part...,ES,56,https://www.camara.leg.br/internet/deputado/ba...
3,2312874-9,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09 16:44:12,Não,178887,https://dadosabertos.camara.leg.br/api/v2/depu...,Rubens Pereira Júnior,PCdoB,https://dadosabertos.camara.leg.br/api/v2/part...,MA,56,https://www.camara.leg.br/internet/deputado/ba...
4,2312874-9,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09 16:44:13,Sim,204436,https://dadosabertos.camara.leg.br/api/v2/depu...,Isnaldo Bulhões Jr.,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AL,56,https://www.camara.leg.br/internet/deputado/ba...
...,...,...,...,...,...,...,...,...,...,...,...,...
670456,2236186-60,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-10-14 20:51:20,Sim,220685,https://dadosabertos.camara.leg.br/api/v2/depu...,Paulo Litro,UNIÃO,https://dadosabertos.camara.leg.br/api/v2/part...,PR,57,https://www.camara.leg.br/internet/deputado/ba...
670457,2236186-60,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-10-14 20:51:25,Sim,178947,https://dadosabertos.camara.leg.br/api/v2/depu...,Sóstenes Cavalcante,PL,https://dadosabertos.camara.leg.br/api/v2/part...,RJ,57,https://www.camara.leg.br/internet/deputado/ba...
670458,2236186-60,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-10-14 20:51:49,Não,74161,https://dadosabertos.camara.leg.br/api/v2/depu...,Reginaldo Lopes,PT,https://dadosabertos.camara.leg.br/api/v2/part...,MG,57,https://www.camara.leg.br/internet/deputado/ba...
670459,2236186-60,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-10-14 20:52:35,Sim,160527,https://dadosabertos.camara.leg.br/api/v2/depu...,Aguinaldo Ribeiro,PP,https://dadosabertos.camara.leg.br/api/v2/part...,PB,57,https://www.camara.leg.br/internet/deputado/ba...


In [5]:
# Temas de cada votação: votação → proposições afetadas (votacoesProposicoes) → temas (proposicoesTemas).
# Os temas ficam no arquivo do ano de APRESENTAÇÃO da proposição, que costuma ser anterior à votação
# (ex.: um PL de 2015 votado em 2023); por isso baixamos os anos de todas as proposições votadas.
vote_props = pd.concat([download('votacoesProposicoes', year) for year in years], ignore_index=True)
vote_props = vote_props[vote_props['idVotacao'].isin(votes['idVotacao'])]

themes = []
for year in sorted(vote_props['proposicao_ano'].dropna().astype(int).unique()):
    try:
        themes.append(download('proposicoesTemas', year))
    except urllib.error.HTTPError as err:
        print(f'proposicoesTemas-{year}: {err}')
themes = pd.concat(themes, ignore_index=True)

vote_themes = (vote_props[['idVotacao', 'proposicao_uri']]
               .merge(themes[['uriProposicao', 'tema']], left_on='proposicao_uri', right_on='uriProposicao')
               [['idVotacao', 'tema']].drop_duplicates(ignore_index=True))
print(f"{vote_themes['idVotacao'].nunique()} de {votes['idVotacao'].nunique()} votações nominais com tema "
      f"({vote_themes['tema'].nunique()} temas)")

2123 de 2186 votações nominais com tema (30 temas)


In [6]:
# Votações por tema e período (uma votação pode ter vários temas)
vote_themes['dataHoraVoto'] = vote_themes['idVotacao'].map(votes.groupby('idVotacao')['dataHoraVoto'].min())
in_period = lambda df, ini, fim: df[(df['dataHoraVoto'] >= ini) & (df['dataHoraVoto'] < fim)]
votacoes_por_tema = pd.DataFrame({label: in_period(vote_themes, ini, fim).groupby('tema')['idVotacao'].nunique()
                                  for label, (ini, fim) in periods.items()}).fillna(0).astype(int)
votacoes_por_tema = votacoes_por_tema.loc[votacoes_por_tema.sum(axis=1).sort_values(ascending=False).index]
geral = {label: in_period(votes, ini, fim)['idVotacao'].nunique() for label, (ini, fim) in periods.items()}
votacoes_por_tema = pd.concat([pd.DataFrame([geral], index=['Geral']), votacoes_por_tema])
votacoes_por_tema

,2022,2023,2024,2025,2026
Geral,586,446,449,550,155
Finanças Públicas e Orçamento,143,137,153,140,31
Administração Pública,102,82,120,162,35
Direitos Humanos e Minorias,64,84,70,85,31
Economia,65,58,73,36,23
Trabalho e Emprego,88,33,21,41,26
Meio Ambiente e Desenvolvimento Sustentável,45,23,66,59,12
Educação,68,24,32,55,10
Saúde,63,37,25,37,12
Direito Penal e Processual Penal,23,21,49,52,19


In [7]:
# Linhas do tempo: "Geral" (todas as votações) + uma por tema
temas = None             # None = todos os temas; ou, p.ex., ['Saúde', 'Educação']
MIN_VOTACOES = 10        # quadros (tema × período) com menos votações ficam de fora
BACKBONE_FRACTION = 0.3  # fração das arestas mantida pelo filtro de disparidade

In [8]:
def get_total_expenses(deputy_id, start_date, end_date):
    #expenses = []
    total = 0.0
    years = '&'.join(['ano={}'.format(year) for year in range(start_date.year, end_date.year+1)])
    url = "https://dadosabertos.camara.leg.br/api/v2/deputados/{}/despesas?{}&itens=100000&ordem=ASC&ordenarPor=ano".format(deputy_id, years)
    read_all = False
    while not read_all:
        read_all = True
        page = requests.get(url).json()
        if 'dados' not in page:
            break
        total += sum([e['valorDocumento'] for e in page['dados']])
        #this_expenses = [{'tipo': e['tipoDespesa'], 'valor': e['valorDocumento']} for e in page['dados']]
        #expenses.extend(this_expenses)
        for link in page['links']:
            if link['rel'] == 'next':
                url = link['href']
                read_all = False
                break
    return total

def get_deputy_info(deputy_id):
    url = 'https://dadosabertos.camara.leg.br/api/v2/deputados/{}'.format(deputy_id)
    data = requests.get(url).json()['dados']
    ultimoStatus = data['ultimoStatus']
    today = date.today()
    nascimento = pd.to_datetime(data['dataNascimento'])
    idade = today.year - nascimento.year - ((today.month, today.day) < (nascimento.month, nascimento.day))
    return {'sexo': data['sexo'],
            'escolaridade': data['escolaridade'],
            'idade': idade,
            'e-mail': ultimoStatus['email'],
            'situação': ultimoStatus['situacao'],
            'cidade natal': data['municipioNascimento']}

In [9]:
#%% Take care of different names for same deputy
# (atribuição direta na coluna: `votes[col].loc[...] = ` e `inplace=True` não alteram `votes` no pandas 3)
votes['deputado_nome'] = votes.groupby('deputado_id')['deputado_nome'].transform(lambda nomes: sorted(nomes.unique())[0])

#%% Partidos que mudaram de nome
votes['deputado_siglaPartido'] = votes['deputado_siglaPartido'].replace(
    {'PMDB': 'MDB', 'PRB': 'REPUBLICANOS', 'PR': 'PL', 'PATRIOTA': 'PATRI', 'PPS': 'CIDADANIA'})

# Logo dos partidos. A API da Câmara não tem o logo dos partidos novos ou renomeados (dá 404),
# então para eles usamos as imagens da Wikipédia / Wikimedia Commons.
LOGOS = {
    'PL': 'https://upload.wikimedia.org/wikipedia/commons/1/12/Partido_Liberal_%28Brazil%29_logo.svg',
    'UNIÃO': 'https://upload.wikimedia.org/wikipedia/commons/7/73/Uni%C3%A3o_Brasil_logo.svg',
    'REPUBLICANOS': 'https://upload.wikimedia.org/wikipedia/pt/0/0d/Republicanos_logo.png',
    'MDB': 'https://logodownload.org/wp-content/uploads/2018/04/mdb-logo-partido.png',
    'SOLIDARIEDADE': 'https://upload.wikimedia.org/wikipedia/commons/f/fe/Logomarca_do_Partido_Solidariedade.png',
    'CIDADANIA': 'https://upload.wikimedia.org/wikipedia/commons/d/d7/Logo_do_Cidadania_23.png',
    'NOVO': 'https://upload.wikimedia.org/wikipedia/commons/b/b5/Novo30_AOC.png',
    'PATRIOTA': 'https://upload.wikimedia.org/wikipedia/commons/f/f9/Logomarca_do_Partido_Patriota.png',
    'PRD': 'https://upload.wikimedia.org/wikipedia/commons/d/dc/Partido_Renova%C3%A7%C3%A3o_Democr%C3%A1tica_logo.svg',
    'MISSÃO': 'https://upload.wikimedia.org/wikipedia/pt/4/47/MISS%C3%83O_logo.png',
    'DC': 'https://upload.wikimedia.org/wikipedia/commons/0/08/Logomarca_Democracia_Crist%C3%A3.png',
    'S.PART.': None,  # sem partido
}
LOGOS['PATRI'] = LOGOS['PATRIOTA']

def get_json(url, tries=5):
    """GET na API da Câmara com novas tentativas (ela devolve erro 504 com frequência)."""
    for attempt in range(tries):
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError):
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))

party_logo = {}
for p in votes['deputado_uriPartido'].dropna().unique():
    dados = get_json(p)['dados']
    party_logo[p] = LOGOS[dados['sigla']] if dados['sigla'] in LOGOS else dados['urlLogo']

#all_data['deputado_siglaPartido'].replace('PPL', np.nan, inplace=True) # PPL foi incorporado
#all_data['deputado_siglaPartido'].replace('PRP', np.nan, inplace=True) # PRP foi incorporado
#all_data['deputado_siglaPartido'].replace('PHS', np.nan, inplace=True) # PHS foi incorporado
# all_data = pd.merge(all_data, motions_themes, on="idVotacao", how="inner")
#all_data['deputado_siglaPartido'].fillna('S.PART.', inplace=True)
# all_data.groupby('idVotacao')['voto'].count()

#all_data.to_csv('votos_{}_to_{}.csv'.format(start_date, end_date), index=False)

In [10]:
# Cores dos partidos: cada partido tem uma cor própria, tirada do logo sempre que possível.
# Usa a cor dominante do logo; se ela ficar parecida demais com a de outro partido, usa uma variação
# dela (mais clara/escura) ou uma cor secundária do logo. Partidos com mais deputados têm prioridade.
sigla_uri = votes.dropna(subset=['deputado_siglaPartido']).groupby('deputado_siglaPartido')['deputado_uriPartido'].last()
sigla_logo = {sigla: party_logo.get(uri) for sigla, uri in sigla_uri.items()}
cores_dos_logos = graph_viz.logo_palette(sigla_logo, cache='data/logo_colors.json')
party_colors = graph_viz.distinct_colors(
    cores_dos_logos,
    weights=votes.groupby('deputado_siglaPartido')['deputado_id'].nunique().to_dict(),
    neutral=['S.PART.'])  # sem partido: cinza
# party_colors['PSOL'] = '#f2d300'  # para fixar a cor de um partido à mão
graph_viz.color_table(party_colors, sigla_logo)

Chave,Claro,Escuro,Cores,Imagem
AVANTE,,,#cf776a #cf776a,
CIDADANIA,,,#ee008d #fd2599,
DC,,,#0080ee #68cbff,
DEM,,,#00a54b #00a54b,
MDB,,,#dfbb00 #dfbb00,
MISSÃO,,,#9f8b40 #9f8b40,
NOVO,,,#d8871d #d8871d,
PATRI,,,#375639 #789387,
PCdoB,,,#ac4955 #ac4955,
PDT,,,#7166c4 #887edf,


## Graph Construction

In [11]:
def generalized_similarity(m, min_eps=0.001, max_iter=1000):
    """ Balázs Kovács, "A generalized model of relational similarity," Social Networks, 32(3), July 2010, pp. 197–211
        Based on: https://github.com/dzinoviev/generalizedsimilarity
    """
    arcs0 = m - m.mean(axis=1)[:, np.newaxis]
    arcs1 = m.T - m.mean(axis=0)[:, np.newaxis]

    eps = min_eps + 1
    N = np.eye(m.shape[1])

    iters = 0
    while (eps > min_eps and iters < max_iter) or np.isnan(N).any():
        M = arcs0.dot(N).dot(arcs0.T)
        m = np.sqrt(M.diagonal())
        M = ((M / (m+1e-8)).T / (m+1e-8)).T

        Np = arcs1.dot(M).dot(arcs1.T)
        n = np.sqrt(Np.diagonal())
        Np = ((Np / (n+1e-8)).T / (n+1e-8)).T
        eps = np.abs(Np - N).max()
        N = Np

        iters += 1
    return M

def pearson_correlation(m):
    arcs0 = m - m.mean(axis=1)[:, np.newaxis]
    arcs1 = m.T - m.mean(axis=0)[:, np.newaxis]
    M = arcs0.dot(arcs0.T)
    m = np.sqrt(M.diagonal())
    M = ((M / m).T / m).T
    return M

def filter_edges(edges_list, num_nodes, threshold=None, density=0.1):
    edges, weights = [], []
    if threshold is not None:
        for e in edges_list:
            if e[1] >= threshold:
                edges.append(e[0])
                weights.append(e[1])
    else:
        count = int(num_nodes * (num_nodes - 1) * density / 2)
        edges_list.sort(reverse=True, key=lambda e: e[1])
        edges_list = edges_list[:count]
        edges = [e[0] for e in edges_list]
        weights = [e[1] for e in edges_list]
    return edges, weights

def disparity_backbone(graph, fraction=None, alpha=None, weight='weight'):
    """Filtro de disparidade: M. Á. Serrano, M. Boguñá, A. Vespignani, "Extracting the multiscale backbone
    of complex weighted networks," PNAS 106(16), 2009.
    Mesmo cálculo de netbone.disparity + netbone.filters.fraction_filter/threshold_filter, direto no igraph.

    Para cada aresta (i, j): p = min sobre as pontas v com grau k_v > 1 de (1 - w_ij / s_v) ** (k_v - 1),
    onde s_v é a força (soma dos pesos) de v. Mantém a fração `fraction` das arestas com menor p
    (desempate: maior peso) ou, se `alpha` for dado, as arestas com p < alpha. Vértices isolados são removidos.
    """
    w = np.asarray(graph.es[weight], dtype=float)
    k = np.asarray(graph.degree())
    s = np.asarray(graph.strength(weights=weight), dtype=float)
    src, dst = np.asarray(graph.get_edgelist()).T
    def alpha_at(v):
        with np.errstate(divide='ignore', invalid='ignore'):
            return np.where(k[v] > 1, (1 - w / s[v]) ** (k[v] - 1), np.nan)
    p = np.fmin(alpha_at(src), alpha_at(dst))  # fmin ignora NaN (ponta com grau 1)

    if alpha is not None:
        keep = np.flatnonzero(p < alpha)
    else:
        order = np.lexsort((-w, np.where(np.isnan(p), np.inf, p)))  # p crescente, depois peso decrescente
        keep = order[:int(np.ceil(fraction * graph.ecount()))]
    graph = graph.copy()
    graph.es['p_value'] = p.tolist()
    return graph.subgraph_edges(sorted(keep.tolist()), delete_vertices=True)

In [12]:
def vote_matrix(vote_data):
    """Matriz deputados × votações (Sim = 1, Não = -1, demais votos = 0), só com quem votou no recorte."""
    vote_data = vote_data.drop_duplicates(['idVotacao', 'deputado_nome'])  # 1º voto de cada deputado (por data)
    valor = vote_data['voto'].map({'Sim': 1.0, 'Não': -1.0}).fillna(0.0)
    return vote_data.assign(valor=valor).pivot(index='deputado_nome', columns='idVotacao', values='valor').fillna(0.0)

def build_graph(vote_data, fraction=BACKBONE_FRACTION):
    """Grafo de similaridade de votos de um recorte (tema × período); None se houver poucas votações."""
    m = vote_matrix(vote_data)
    if m.shape[1] < MIN_VOTACOES:
        return None
    M = generalized_similarity(m.values)
    i, j = np.triu_indices(len(m), k=1)
    sim = M[i, j]
    pos = sim > 0
    if not pos.any():
        return None
    graph = Graph(n=len(m), edges=np.column_stack([i[pos], j[pos]]).tolist(), directed=False)
    graph.vs['name'] = m.index.tolist()
    graph.es['similarity'] = sim[pos].tolist()
    span = (sim[pos].max() - sim[pos].min()) or 1.0
    graph.es['weight'] = ((sim[pos] - sim[pos].min()) / span).tolist()  # pesos normalizados em [0, 1]

    # Backbone: filtro de disparidade
    graph = disparity_backbone(graph, fraction=fraction, weight='weight')

    # Comunidades (Leiden) e centralidades
    graph.vs['community'] = graph.community_leiden(weights='weight', objective_function='modularity', n_iterations=100).membership
    graph['votações'] = int(m.shape[1])
    graph['modularidade'] = round(graph.modularity(graph.vs['community'], weights='weight'), 3)
    graph.vs['betweenness'] = graph.betweenness()
    graph.vs['closeness'] = graph.closeness()

    # Dados de cada deputado no recorte (último voto: partido e UF da época)
    info = vote_data.groupby('deputado_nome').last().reindex(graph.vs['name'])
    graph.vs['Partido'] = info['deputado_siglaPartido'].tolist()
    graph.vs['UF'] = info['deputado_siglaUf'].tolist()
    graph.vs['Partido URI'] = info['deputado_uriPartido'].tolist()
    graph.vs['Foto'] = info['deputado_urlFoto'].tolist()
    graph.vs['URL'] = info['deputado_uri'].tolist()
    graph.vs['Página'] = [u.replace('dadosabertos.camara.leg.br/api/v2/deputados/', 'www.camara.leg.br/deputados/')
                          if isinstance(u, str) else None for u in graph.vs['URL']]
    return graph

In [13]:
random.seed(42)  # Leiden reprodutível (o igraph usa o `random` do Python)
lista_temas = temas if temas is not None else votacoes_por_tema.index.drop('Geral').tolist()
recortes = {'Geral': votes}
for tema in lista_temas:
    recortes[tema] = votes[votes['idVotacao'].isin(vote_themes.loc[vote_themes['tema'] == tema, 'idVotacao'])]

timelines = {}
for nome, vote_data in recortes.items():
    frames = {}
    for label, (ini, fim) in periods.items():
        graph = build_graph(in_period(vote_data, ini, fim))
        if graph is not None:
            frames[label] = graph
    if frames:
        # Rótulos de comunidade estáveis no tempo ("Comunidade 1" de 2023 é a que mais se sobrepõe à de 2022)
        timelines[nome] = graph_viz.align_labels(frames, attr='community', new_attr='Comunidade', label='Comunidade {}')
        print(f"{nome}: " + ', '.join(f"{p} ({g.vcount()} dep., {g.ecount()} arestas)" for p, g in frames.items()))

Geral: 2022 (554 dep., 28641 arestas), 2023 (543 dep., 28191 arestas), 2024 (566 dep., 27999 arestas), 2025 (551 dep., 25966 arestas), 2026 (567 dep., 27359 arestas)
Finanças Públicas e Orçamento: 2022 (553 dep., 27687 arestas), 2023 (554 dep., 28985 arestas), 2024 (571 dep., 28713 arestas), 2025 (549 dep., 26695 arestas), 2026 (508 dep., 31251 arestas)
Administração Pública: 2022 (537 dep., 29063 arestas), 2023 (549 dep., 27791 arestas), 2024 (528 dep., 27653 arestas), 2025 (526 dep., 24346 arestas), 2026 (546 dep., 26803 arestas)
Direitos Humanos e Minorias: 2022 (551 dep., 29094 arestas), 2023 (495 dep., 29208 arestas), 2024 (514 dep., 31116 arestas), 2025 (536 dep., 25066 arestas), 2026 (559 dep., 27501 arestas)
Economia: 2022 (551 dep., 26909 arestas), 2023 (526 dep., 28204 arestas), 2024 (561 dep., 27341 arestas), 2025 (538 dep., 25313 arestas), 2026 (561 dep., 27802 arestas)
Trabalho e Emprego: 2022 (525 dep., 27161 arestas), 2023 (506 dep., 28320 arestas), 2024 (559 dep., 29502

In [14]:
# Modularidade de cada quadro (quanto maior, mais o plenário se divide em blocos que votam juntos)
pd.DataFrame({nome: {p: g['modularidade'] for p, g in frames.items()} for nome, frames in timelines.items()}).T

,2022,2023,2024,2025,2026
Geral,0.518,0.618,0.659,0.561,0.633
Finanças Públicas e Orçamento,0.597,0.585,0.603,0.641,0.583
Administração Pública,0.638,0.624,0.642,0.508,0.579
Direitos Humanos e Minorias,0.631,0.583,0.517,0.351,0.540
Economia,0.598,0.654,0.403,0.647,0.642
Trabalho e Emprego,0.625,0.591,0.601,0.521,0.606
Meio Ambiente e Desenvolvimento Sustentável,0.648,0.643,0.642,0.562,0.525
Educação,0.599,0.545,0.604,0.637,0.656
Saúde,0.633,0.489,0.454,0.586,0.620
Direito Penal e Processual Penal,0.638,0.585,0.619,0.569,0.605


## Visualization

In [15]:
# Gera um HTML autocontido (Sigma.js + Graphology + ForceAtlas2) e exibe aqui no notebook.
# Embaixo fica a linha do tempo: escolha o tema e mude o período (ou "▶ Animar"); o layout se ajusta ao vivo.
# O arquivo camara.html abre direto em qualquer navegador e pode ser publicado (ex.: GitHub Pages).
graph_viz.show(timelines,
               path='camara.html',
               title='Câmara dos Deputados',
               subtitle='Similaridade de votos nominais',
               logos=party_logo,       # logos dos partidos no painel de detalhes
               colors={'Partido': party_colors},  # uma cor por partido (tabela acima)
               layout_seconds=4,       # segundos de ForceAtlas2 ao abrir
               transition_seconds=3,   # segundos de ForceAtlas2 a cada troca de período ou tema
               play_seconds=4,         # intervalo entre períodos no "▶ Animar"
               # fa2_settings={'gravity': 0.5, 'linLogMode': True},  # parâmetros do ForceAtlas2: ajuste ao vivo em "⚙ Parâmetros" e copie de lá
               height=820)

Visualização salva em /home/lucas/vote-network/camara.html (8.0 MB) — abra em qualquer navegador.
